<a href="https://colab.research.google.com/github/MoonlitTech/ddos-detection/blob/main/ddos_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd

df = pd.read_csv("ddos-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

print(df.shape)              # rows and columns
print(list(df.columns))      # all column names
print(df[df.columns[-1]].value_counts())   # label counts

(225745, 79)
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count'

In [21]:
import os; print(os.path.getsize("ddos-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"))

77123859


# BLANK SPACE

# COMP 365 Individual Project - DDoS Detection
DDoS Detection: Rule-Based Baseline

**Goal:**
Classify network flows as either benign or DDoS using a simple
threshold rule based on packets per second.

This is the baseline that will later be compared with machine-learning
models such as Logistic Regression, Decision Tree, and Random Forest.

EVALUATION MEASURES:
1. Precision: How often predicted DDoS attacks are actually DDoS.
2. Recall: How many actual DDoS attacks are correctly detected.
3. Inference time per flow: Average time needed to make a prediction.

This code uses only built-in Python libraries.

**AI disclosure: Document any AI assistance actually used in your
project in your README, following the assignment requirements.**

# 1. Import libraries and set up the dataset

* csv reads the dataset file.
* Math helps identify invalid numeric values such as infinity.
* Random shuffles the data before splitting it.
* Time measures how long predictions take.

In [22]:
import csv
import math
import random
import time

# Google Colab stores uploaded files in /content by default.
CSV_PATH = "/ddos-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

# Using a fixed seed makes the random split repeatable.
random.seed(42)

# 2. Load and clean the data
Each usable row stores two values:
* Flow Packets/s: the number of packets per second.
* Label: 1 for DDoS and 0 for any other label.
<br>

Features:
* Exact duplicate rows are skipped.
* Rows with unreadable or invalid packet rates are skipped.
* This code uses one feature only, so it is a simple baseline.

In [23]:
rows = []
seen = set()

with open(CSV_PATH, newline="") as f:
    for r in csv.DictReader(f):

        # Remove extra spaces from column names.
        r = {k.strip(): v for k, v in r.items()}

        # Skip rows that are exact duplicates.
        key = tuple(r.items())
        if key in seen:
            continue
        seen.add(key)

        # Convert the packet rate into a number.
        try:
            rate = float(r["Flow Packets/s"])
        except (ValueError, TypeError):
            continue

        # Skip missing or infinite numeric values.
        if not math.isfinite(rate):
            continue

        # Convert the label into a binary value:
        # 1 means DDoS; 0 means any other label.
        if r["Label"] == "DDoS":
            label = 1
        else:
            label = 0

        rows.append((rate, label))

print("Flows kept:", len(rows))

Flows kept: 223082


# 3. SPLIT THE DATA INTO TRAINING AND TEST SETS
70% of the data is used to select the threshold.
The remaining 30% is held back to evaluate the final rule.
The test set should not be used to choose the threshold, because that would make the evaluation less independent.

In [24]:
random.shuffle(rows)

cut = int(len(rows) * 0.7)
train, test = rows[:cut], rows[cut:]

print("Training flows:", len(train))
print("Test flows:", len(test))

Training flows: 156157
Test flows: 66925


# 4. DEFINE THE PREDICTION RULE AND EVALUATION MEASURES
The rule predicts DDoS based on whether the packet rate is above or below a selected threshold.

Precision = correct DDoS predictions / all predicted DDoS flows.

Recall = correctly detected DDoS flows / all actual DDoS flows.

F1 is also calculated internally to choose the threshold. It will not be included in the final printed evaluation results.

In [25]:
def predict_rates(rates, cutoff, direction):
    # Return 1 for predicted DDoS and 0 for predicted non-DDoS
    if direction == ">":
        return [1 if rate > cutoff else 0 for rate in rates]

    return [1 if rate < cutoff else 0 for rate in rates]

def calculate_scores(labels, predictions):
    # Count correct and incorrect predictions
    tp = fp = tn = fn = 0

    for label, guess in zip(labels, predictions):
        if guess == 1 and label == 1:
            tp += 1  # Correctly detected DDoS
        elif guess == 1 and label == 0:
            fp += 1  # Benign flow incorrectly flagged
        elif guess == 0 and label == 0:
            tn += 1  # Correctly identified benign flow
        else:
            fn += 1  # DDoS attack missed

    # Calculate precision and recall
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    # F1 is used only to select the baseline threshold
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )

    return precision, recall, f1

# 5. SELECT THE THRESHOLD USING TRAINING DATA ONLY
Try different packet-rate thresholds from the training set.
Test both possible rules:
* Predict DDoS when the rate is greater than the threshold.
* Predict DDoS when the rate is less than the threshold.
* The threshold with the highest training F1-score is selected.
* The test set is not used during this selection process.

In [26]:
sorted_rates = sorted(rate for rate, _ in train)

# Store the best score, threshold, and rule direction.
best = (-1.0, None, None)

for percent in range(1, 100):
    cutoff = sorted_rates[int(len(sorted_rates) * percent / 100)]

    for direction in (">", "<"):
        train_rates = [rate for rate, _ in train]
        train_labels = [label for _, label in train]

        train_predictions = predict_rates(
            train_rates, cutoff, direction
        )

        f1 = calculate_scores(
            train_labels, train_predictions
        ) [2]

        if f1 > best[0]:
            best = (f1, cutoff, direction)

# Keep the selected threshold and rule direction.
_, cutoff, direction = best

print(
    f"Selected rule: predict DDoS if Flow Packets/s "
    f"{direction} {cutoff}"
)


Selected rule: predict DDoS if Flow Packets/s < 2941.176471


# 6. EVALUATE THE RULE ON THE TEST SET
* Apply the selected rule to the test data.
* Measure the time needed to make predictions.
* Divide total prediction time by the number of test flows
* to estimate the average inference time per flow.
* Precision and recall are calculated after timing, so the
* metric calculations do not inflate the prediction time.

In [27]:
test_rates = [rate for rate, _ in test]
test_labels = [label for _, label in test]

# Time only the prediction operation.
start = time.perf_counter()

test_predictions = predict_rates(
    test_rates, cutoff, direction
)

elapsed = time.perf_counter() - start

# Calculate evaluation measures after timing.
precision, recall, _ = calculate_scores(
    test_labels, test_predictions
)

# Calculate average prediction time per flow.
inference_time_per_flow = (
    elapsed / len(test) if test else 0.0
)

# 7. DISPLAY THE FINAL EVALUATION RESULTS
* These are the three measures used to evaluate this baseline.
* Use the actual output values when reporting project results.

In [28]:
print(f"Test precision: {precision:.4f}")
print(f"Test recall: {recall:.4f}")

print(
    f"Inference time per flow: "
    f"{inference_time_per_flow:.8f} seconds"
)

print(
    f"Inference time per flow: "
    f"{inference_time_per_flow * 1000:.6f} milliseconds"
)

Test precision: 0.6449
Test recall: 0.9989
Inference time per flow: 0.00000014 seconds
Inference time per flow: 0.000137 milliseconds
